# AgeBand — Interactive Demo

**AgeBand** is a passive age-band inference system: it reads how a user writes and
what they talk about, maintains a live estimate of their age band
(`child / teen / adult / unknown`), and emits a `safety_posture` the host product
can act on — without ever touching the reply path.

The model *estimates*; deterministic Python *decides*. Confidence is always
computed from evidence, never from the LLM; inferred bands never persist.

---

### Runs locally **or** on a remote AMD GPU box (single port)

This notebook starts **one** server that serves both the API and the UI on a
single port (`SERVER_PORT`) — one origin, no CORS, no reverse proxy.

- **Local:** open `http://localhost:<SERVER_PORT>/`
- **Remote (AMD droplet, public IP):** set `SERVER_PORT` to an **open inbound port**
  and `AGEBAND_PUBLIC_HOST` to the droplet's public IP, then open
  `http://<public-ip>:<SERVER_PORT>/`

**LLM mode** (default) expects a running vLLM OpenAI endpoint
(`LOCAL_API_BASE`, default `http://localhost:8001/v1`, model `google/gemma-3-27b-it`).
Set `AGEBAND_INFERENCE_MODE=deterministic` to run with no GPU / no endpoint.

In [ ]:
# ── USER SETTINGS — EDIT THESE ────────────────────────────────────────────────
# Set here BEFORE any other cell runs. These override the defaults read in the
# Configuration cell below, so edit-then-"Run All" works with no other changes.
import os

# 1) Port the AgeBand app (combined API + UI) listens on.
#    Must be an OPEN inbound port on the droplet (e.g. `sudo ufw allow <port>/tcp`).
os.environ["SERVER_PORT"] = "8080"

# 2) Hugging Face token — required to download GATED models (Gemma / Llama /
#    Mistral). Create one at https://huggingface.co/settings/tokens and accept
#    the model's license on its HF page first. Leave "" if your model is ungated.
os.environ["HF_TOKEN"] = ""

# 3) (Optional) Public IP/hostname of the droplet, so the printed UI links are
#    clickable from your laptop. Leave "" for a local run.
os.environ["AGEBAND_PUBLIC_HOST"] = ""

_tok = os.environ["HF_TOKEN"]
print(f"SERVER_PORT         = {os.environ['SERVER_PORT']}")
print(f"HF_TOKEN            = {'set (' + str(len(_tok)) + ' chars)' if _tok else 'NOT set'}")
print(f"AGEBAND_PUBLIC_HOST = {os.environ['AGEBAND_PUBLIC_HOST'] or '(local)'}")


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Check Python version and install any missing deps without re-running pip every time.
import sys
print(f"Python {sys.version}")

# Verify node/npm available for the UI build step.
import shutil
node_ok = shutil.which("node") is not None
npm_ok  = shutil.which("npm")  is not None
print(f"node: {'✓' if node_ok else '✗ NOT FOUND — UI build will be skipped'}")
print(f"npm:  {'✓' if npm_ok  else '✗ NOT FOUND — UI build will be skipped'}")

# Install missing notebook deps (idempotent — skips if already importable).
def _ensure_importable(package: str, pip_name: str = "") -> None:
    try:
        __import__(package)
    except ImportError:
        import subprocess as _sp
        _sp.run([sys.executable, "-m", "pip", "install", "-q",
                 pip_name or package], check=True)

_ensure_importable("requests")
_ensure_importable("pandas")
_ensure_importable("IPython")
_ensure_importable("jinja2")
print("\n✓ All notebook dependencies available")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
# Everything is env-overridable so the SAME notebook runs locally and on the droplet.
import os, sys
from pathlib import Path

# Find the repo root regardless of where this .ipynb file lives.
def _find_repo_root() -> Path:
    env = os.environ.get("AGEBAND_DIR", "").strip()
    if env and (Path(env) / "src/orchestration/api.py").exists():
        return Path(env).resolve()
    here = Path(os.getcwd())
    for cand in (here, *here.parents):
        if (cand / "src/orchestration/api.py").exists():
            return cand.resolve()
    if (Path.home() / "ageBand/src/orchestration/api.py").exists():
        return (Path.home() / "ageBand").resolve()
    return here.resolve()

REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

# Mode: 'llm' (needs a vLLM endpoint) or 'deterministic' (no GPU).
INFERENCE_MODE = os.environ.get("AGEBAND_INFERENCE_MODE", "llm").strip().lower()

# Single combined port for API + UI (override for a remote open port, e.g. 30000).
SERVER_PORT = int(os.environ.get("SERVER_PORT", "8080"))

# LLM endpoint — defaults match the vLLM we serve on the MI300X droplet.
os.environ.setdefault("LOCAL_API_BASE", "http://localhost:8001/v1")
os.environ.setdefault("LOCAL_API_KEY", "EMPTY")
os.environ.setdefault("LOCAL_MODEL",     "google/gemma-3-27b-it")
os.environ.setdefault("EXTRACTOR_MODEL", os.environ["LOCAL_MODEL"])
os.environ.setdefault("ESTIMATOR_MODEL", os.environ["LOCAL_MODEL"])

# Runtime flags (ROCm vLLM compat + telemetry + single-process UI).
os.environ["AGEBAND_INFERENCE_MODE"] = INFERENCE_MODE
os.environ["AGEBAND_SERVE_UI"] = "1"                        # one process serves UI + API
os.environ.setdefault("AGEBAND_NO_RESPONSE_FORMAT", "1")   # broken ROCm xgrammar → omit response_format
os.environ.setdefault("GUIDED_DECODING_ENABLED", "")       # off on ROCm
os.environ.setdefault("SKIP_AMD_CHECK", "true" if INFERENCE_MODE == "deterministic" else "false")
os.environ.setdefault("VLLM_METRICS_URL", os.environ["LOCAL_API_BASE"].replace("/v1", "") + "/metrics")

from scripts.notebook_server_utils import detect_public_host
PUBLIC_HOST = detect_public_host()
UI_URL = f"http://{PUBLIC_HOST}:{SERVER_PORT}/"

print(f"Repo root   : {REPO_ROOT}")
print(f"Mode        : {INFERENCE_MODE}")
print(f"Server port : {SERVER_PORT}   (API + UI on one origin)")
if INFERENCE_MODE != "deterministic":
    print(f"LLM endpoint: {os.environ['LOCAL_API_BASE']}   model={os.environ['LOCAL_MODEL']}")
print(f"Browser URL : {UI_URL}")
print("  Remote tip: export AGEBAND_PUBLIC_HOST=<public-ip> and SERVER_PORT=<open-port> before launching Jupyter.")

In [ ]:
# ── Step 3.5: Auto-start vLLM (only when inference mode is 'llm') ──────────
# Logic:
#   1. INFERENCE_MODE != 'llm'          → skip (deterministic mode)
#   2. LOCAL_API_BASE already reachable  → use as-is (never restart a server we don't own)
#   3. VLLM_HOST points to this machine  → auto-start vLLM
#   4. VLLM_HOST is a remote machine     → can't start it here — print instructions
#
# Dual-model support: if EXTRACTOR_MODEL / ESTIMATOR_MODEL differ from LOCAL_MODEL
# the server is started with --served-model-name so ONE vLLM process responds to
# ALL three model names.  No second process needed.
#
# On a CPU-only machine set VLLM_ALLOW_CPU = True to override the safety guard.
VLLM_ALLOW_CPU = False

# Track whether THIS run started vLLM so the cleanup cell only stops what it owns.
_vllm_proc   = None
_vllm_owned  = False  # True only if we started it; never teardown a pre-existing server.

if INFERENCE_MODE != "llm":
    print(f"INFERENCE_MODE={INFERENCE_MODE!r} → skipping vLLM auto-start (deterministic mode)")
else:
    import threading, urllib.request as _urlreq
    from urllib.parse import urlparse as _urlparse
    from scripts.notebook_server_utils import detect_amd_gpu, is_local_host, start_vllm, wait_for_url

    _base   = os.environ["LOCAL_API_BASE"].rstrip("/")
    _parsed = _urlparse(_base)
    VLLM_HOST = _parsed.hostname or "localhost"
    VLLM_PORT = _parsed.port or 8001
    _models_url = f"{_base}/models"

    # Collect the model names this run needs so we can register them all as
    # --served-model-name aliases on one vLLM process.
    _model     = os.environ.get("LOCAL_MODEL", "")
    _extractor = os.environ.get("EXTRACTOR_MODEL", _model)
    _estimator = os.environ.get("ESTIMATOR_MODEL", _model)
    # Deduplicated, order preserved (LOCAL_MODEL first → that's what vLLM loads)
    _seen_names: set[str] = set()
    _all_names: list[str] = []
    for _nm in [_model, _extractor, _estimator]:
        if _nm and _nm not in _seen_names:
            _all_names.append(_nm)
            _seen_names.add(_nm)

    def _vllm_up() -> bool:
        try:
            with _urlreq.urlopen(_models_url, timeout=3) as r:
                return r.status == 200
        except Exception:
            return False

    if _vllm_up():
        # ── Case 2: already running — reuse it ────────────────────────────
        print(f"✓ vLLM already running at {VLLM_HOST}:{VLLM_PORT} — using it as-is")
        _vllm_owned = False

    elif not is_local_host(VLLM_HOST):
        # ── Case 4: remote machine ────────────────────────────────────────
        raise RuntimeError(
            f"vLLM at {VLLM_HOST}:{VLLM_PORT} is unreachable and VLLM_HOST points to a\n"
            f"different machine.  This notebook can only start vLLM on the local box.\n"
            f"  → SSH into {VLLM_HOST} and start vLLM there, then re-run this cell.\n"
            f"  → Or set LOCAL_API_BASE to a local endpoint (e.g. http://localhost:8001/v1)."
        )

    else:
        # ── Case 3: local host — check GPU and start ──────────────────────
        _has_gpu = detect_amd_gpu()
        print(f"AMD GPU (ROCm) detected: {_has_gpu}")

        if not _has_gpu and not VLLM_ALLOW_CPU:
            print()
            print("=" * 64)
            print("⚠️  No AMD GPU detected — vLLM on CPU will be extremely slow")
            print("=" * 64)
            print()
            print("Options:")
            print("  A) Set VLLM_ALLOW_CPU = True in this cell and re-run")
            print("     (only practical for tiny models like gemma-3-4b-it)")
            print("  B) Set INFERENCE_MODE = 'deterministic' in the config cell")
            print("     to skip the LLM entirely (instant, no GPU)")
            print("=" * 64)
            raise RuntimeError(
                "No AMD GPU. Set VLLM_ALLOW_CPU=True or INFERENCE_MODE='deterministic'."
            )
        elif not _has_gpu:
            print("⚠️  VLLM_ALLOW_CPU=True — starting on CPU (this will be slow)")

        # HF_TOKEN check for gated models (Gemma / Llama require license acceptance).
        _model_lc  = _model.lower()
        _gated     = any(g in _model_lc for g in ("gemma", "llama", "mistral"))
        _hf_token  = os.environ.get("HF_TOKEN", "")

        if _gated and not _hf_token:
            _hf_url = f"https://huggingface.co/{_model}"
            raise RuntimeError(
                f"{_model!r} is HuggingFace-gated and requires license acceptance.\n"
                f"  1. Accept the license at {_hf_url}\n"
                f"  2. Set HF_TOKEN before re-running:\n"
                f"       import os; os.environ['HF_TOKEN'] = 'hf_...'\n"
                f"  Or set LOCAL_MODEL to an ungated model."
            )

        _alias_label = (
            f" (aliases: {', '.join(_all_names[1:])})" if len(_all_names) > 1 else ""
        )
        print(f"\nStarting vLLM serving {_model!r}{_alias_label} on port {VLLM_PORT} …")
        if len(_all_names) > 1:
            print(f"  → --served-model-name registers all {len(_all_names)} names on one process")
        print("(First run downloads model weights — several GB. Watch progress below.)")
        print(f"Polling {_models_url} …\n")

        _vllm_proc = start_vllm(
            _model,
            port=VLLM_PORT,
            hf_token=_hf_token,
            served_model_names=_all_names,
        )
        _vllm_owned = True
        print(f"vLLM process started (PID {_vllm_proc.pid})")

        # Stream startup / download progress in a background thread while polling.
        def _stream(proc, max_lines: int = 60) -> None:
            stdout = getattr(proc._proc, "stdout", None)
            if stdout is None:
                return
            count = 0
            try:
                for raw in stdout:
                    line = raw.decode("utf-8", errors="replace").rstrip()
                    if line:
                        print(f"  [vllm] {line}")
                        count += 1
                    if count >= max_lines:
                        print("  [vllm] … (truncated; waiting for /v1/models to respond) …")
                        break
            except Exception:
                pass

        _t = threading.Thread(target=_stream, args=(_vllm_proc,), daemon=True)
        _t.start()

        if wait_for_url(_models_url, timeout=600, interval=5):
            print(f"\n✓ vLLM is ready at {_base}")
        else:
            print("\n✗ vLLM did not become ready within 10 minutes.")
            print("  Check: model name, HF_TOKEN, available VRAM, disk space.")
            raise RuntimeError("vLLM startup timed out — see output above for details")


In [ ]:
# ── Step 3.6: Runtime vLLM model verification ────────────────────────────────
# Fetch /v1/models and confirm what vLLM is actually serving matches the
# configured model names.  Catches drift that a static grep cannot: e.g.
# vLLM serving a stale cached model despite a fresh config change.
# Skips gracefully in deterministic mode (no vLLM endpoint expected).
import urllib.request as _urlreq2, json as _json2

if INFERENCE_MODE != "llm":
    print("INFERENCE_MODE=deterministic — skipping vLLM model verification")
else:
    _vbase = os.environ["LOCAL_API_BASE"].rstrip("/")
    _murl  = f"{_vbase}/models"

    _served: list[str] = []
    try:
        with _urlreq2.urlopen(_murl, timeout=5) as _r:
            _d = _json2.loads(_r.read())
            _served = [m["id"] for m in _d.get("data", [])]
    except Exception as _e:
        print(f"⚠️  Cannot reach {_murl}: {_e}")
        print("   (If vLLM is still starting, wait a moment and re-run this cell)")

    _local_m = os.environ.get("LOCAL_MODEL", "")
    _ext_m   = os.environ.get("EXTRACTOR_MODEL", _local_m)
    _est_m   = os.environ.get("ESTIMATOR_MODEL", _local_m)

    def _model_covered(name: str, served: list[str]) -> bool:
        return any(s == name or s.endswith(name.split("/")[-1]) for s in served)

    print(f"{'Configured variable':<22}  {'Model name':<38}  Served?")
    print("-" * 70)
    for lbl, val in [("LOCAL_MODEL", _local_m),
                      ("EXTRACTOR_MODEL", _ext_m),
                      ("ESTIMATOR_MODEL", _est_m)]:
        if _served:
            ok = _model_covered(val, _served)
            flag = "✓" if ok else "✗ NOT SERVED"
        else:
            flag = "(vLLM not reached)"
        print(f"  {lbl:<20}  {val:<38}  {flag}")

    if _served:
        print(f"\nvLLM is serving: {', '.join(_served)}")

        _all_needed = {m for m in [_local_m, _ext_m, _est_m] if m}
        _missing = {m for m in _all_needed if not _model_covered(m, _served)}
        if _missing:
            print()
            print(f"⚠️  These configured models are NOT served by vLLM: {', '.join(_missing)}")
            print("   Agent requests for those models will fail with 'model not found'.")
            print("   Fix: either update the model env vars to match what vLLM is serving,")
            print("   or restart vLLM with the correct model.")
        else:
            print("\n✓ All configured models are covered by the running vLLM instance")
    else:
        print("\n(Model verification skipped — vLLM endpoint not reachable)")


In [ ]:
# ── Start the combined AgeBand server (API + UI, one port) ────────────────
from scripts.notebook_server_utils import start_agent, wait_for_url

AGENT_HEALTH_URL = f"http://127.0.0.1:{SERVER_PORT}/health"
_SERVER_LOG = str(REPO_ROOT / "ui_agent.log")

# Clean up a stale process from a previous run.
try:
    _agent_proc  # type: ignore[name-defined]
    if _agent_proc.is_running():  # type: ignore[name-defined]
        print("Stopping previous server …")
        _agent_proc.stop()  # type: ignore[name-defined]
except NameError:
    pass

_agent_proc = start_agent(
    port=SERVER_PORT,
    inference_mode=INFERENCE_MODE,
    repo_root=REPO_ROOT,
    host="0.0.0.0",       # bind all interfaces so a remote browser can reach it
    serve_ui=True,        # same process serves the built UI at /
    log_path=_SERVER_LOG,
)
print(f"Server started (PID {_agent_proc.pid}) — polling {AGENT_HEALTH_URL} …")

if wait_for_url(AGENT_HEALTH_URL, timeout=60):
    print("✓ Server is up (API + UI)")
else:
    print("✗ Server did not become healthy within 60 s. Log tail:")
    print(_agent_proc.tail(40))
    raise RuntimeError("Server startup failed — is the vLLM endpoint reachable? (LLM mode)")

In [ ]:
# ── Ensure the UI build exists (served by the combined server at /) ─────────
import subprocess, requests
UI_DIR  = REPO_ROOT / "src" / "ui"
UI_DIST = UI_DIR / "dist"

if not UI_DIST.exists():
    if node_ok and npm_ok:
        print("dist/ missing — building UI (npm install + build) …")
        if not (UI_DIR / "node_modules").exists():
            subprocess.run(["npm", "install"], cwd=UI_DIR, check=True)
        subprocess.run(["npm", "run", "build"], cwd=UI_DIR, check=True)
        print("Build done — re-run the server cell above so it picks up dist/.")
    else:
        print("⚠️  dist/ missing and node/npm not found. The UI will 404, but the")
        print("    API + the scenario/roster cells below still work. Build dist/ on a")
        print("    machine with Node (npm run build in src/ui), or just use the flows.")
else:
    r = requests.get(f"http://127.0.0.1:{SERVER_PORT}/", timeout=5)
    if r.status_code == 200:
        print(f"✓ UI served by the combined server: HTTP 200, {len(r.content)} bytes")
    else:
        print(f"⚠️  UI index returned HTTP {r.status_code} — check {_SERVER_LOG}")

## AgeBand UI

The combined server serves the UI at the **Browser URL** printed by the config cell.

> **Open it in a new browser tab** — this is the reliable path, especially on a
> remote box. The link is printed live in the next cell.
>
> The inline iframe below can be blocked when JupyterLab is served over **HTTPS**
> but the UI over **HTTP** (mixed-content), or by localhost-iframe policies. If the
> frame is blank, use the tab link.

Use the **Session** tab to send turns (band / confidence / posture update live) and
the **Roster** tab to replay the bundled synthetic Discord export.

In [ ]:
# ── Link + inline UI (dark theme) ─────────────────────────────────
# The notebook always loads the UI with ?theme=dark so the inline frame renders
# the instrument-panel dark palette.  Opening the plain URL (no query param)
# in a new tab still shows the default light theme — the theme is opt-in.
from IPython.display import IFrame, display, Markdown

_dark_url = UI_URL.rstrip("/") + "/?theme=dark"
display(Markdown(f"### ▶︎ Open the UI in a new tab: [{_dark_url}]({_dark_url})"))
try:
    display(IFrame(src=_dark_url, width=1100, height=720))
except Exception as exc:  # noqa: BLE001
    display(Markdown(f"_(inline iframe unavailable: {exc} — use the tab link above)_"))

---
## Scenario walkthroughs

The four cells below replay the same scenarios covered by `tests/e2e/` directly
against the running agent, printing `band`, `confidence`, `posture`, and a short
evidence/trace summary for each turn.

In **deterministic mode** the keyword extractor and rule estimator run (no LLM).
In **LLM mode** the actual model handles extraction and estimation — the safety
logic (gate, confidence, policy, posture) is always deterministic regardless.

In [ ]:
# ── Shared request helpers ──────────────────────────────────────
import json, requests

AGENT_BASE = f"http://127.0.0.1:{SERVER_PORT}"

def send_turn(session_id: str, text: str, turn_number: int) -> dict:
    """POST a single turn and return the parsed JSON response."""
    resp = requests.post(
        f"{AGENT_BASE}/v1/turn",
        json={"session_id": session_id, "turn_text": text, "turn_number": turn_number},
        timeout=30,   # LLM turns are ~2s on MI300X; headroom for cold cache
    )
    resp.raise_for_status()
    return resp.json()

def print_turn(label: str, text: str, result: dict) -> None:
    """Pretty-print a turn result for the notebook."""
    posture = result.get("posture", {})
    evidence = result.get("evidence", {})
    cues = evidence.get("cues", [])
    print(f"  Turn : {text!r}")
    print(f"  Band : {result.get('band', '?')}")
    print(f"  Conf : {result.get('confidence', 0):.2f}")
    print(f"  Post : {posture.get('level', '?')}  flags={posture.get('flags', {})}")
    if cues:
        cue_summary = ", ".join(f"{c['type']}:{c['value'][:30]}" for c in cues[:3])
        print(f"  Cues : {cue_summary}{'…' if len(cues) > 3 else ''}")

In [ ]:
# ── Scenario 1: Clear Adult ────────────────────────────────────────────────────
# Expectation: standard posture — an adult who writes about adult topics
# must NOT be over-restricted.
print("=" * 60)
print("SCENARIO 1 — Clear Adult")
print("=" * 60)
ADULT_TURNS = [
    "I've been following the geopolitical situation with considerable interest.",
    "My MBA thesis on corporate strategy was well received by the committee.",
    "I'm reviewing our Q3 earnings and the macroeconomic outlook looks uncertain.",
]
for i, text in enumerate(ADULT_TURNS, start=1):
    r = send_turn("demo-adult", text, i)
    print(f"\n[Turn {i}]")
    print_turn("adult", text, r)

print("\n✓ Expected: band=adult, posture=standard")

In [ ]:
# ── Scenario 2: Young Teen ─────────────────────────────────────────────────────
# Expectation: elevated posture (caution or restricted) — school/guardian cues
# and a grade-level disclosure are strong signals.
print("=" * 60)
print("SCENARIO 2 — Young Teen")
print("=" * 60)
TEEN_TURNS = [
    "My parents won't let me go to the party. I'm in 8th grade and it's not fair.",
    "omg my math teacher gave SO much homework today, like 3 worksheets",
    "Can you help me write an excuse note for school? My mom is out of town.",
]
for i, text in enumerate(TEEN_TURNS, start=1):
    r = send_turn("demo-teen", text, i)
    print(f"\n[Turn {i}]")
    print_turn("teen", text, r)

print("\n✓ Expected: band=teen, posture=caution or restricted")

In [ ]:
# ── Scenario 3: Ambiguous Adult (Fairness) ─────────────────────────────────────
# Expectation: standard posture — low-signal text must NOT over-restrict.
# An adult who writes tersely (non-native speaker, terse style) looks identical
# to a child on surface metrics; confidence must stay low → standard posture.
print("=" * 60)
print("SCENARIO 3 — Ambiguous Adult  (Fairness test)")
print("=" * 60)
AMBIG_TURNS = [
    "I want to learn more about cooking.",
    "What is a good recipe for pasta?",
    "How long does it take to boil water?",
]
for i, text in enumerate(AMBIG_TURNS, start=1):
    r = send_turn("demo-ambiguous", text, i)
    print(f"\n[Turn {i}]")
    print_turn("ambiguous", text, r)

print("\n✓ Expected: band=unknown, posture=standard  (no evidence → no restriction)")

In [ ]:
# ── Scenario 4: Adversarial — Child Claiming Adult ────────────────────────────
# The crux scenario for the hackathon demo.
#
# A user insists they are an adult, protests age questions, and repeats the claim
# unprompted.  AgeBand's evasion guard should:
#   • Detect the masking pattern (over-insistence, deflection)
#   • Apply the evasion confidence penalty (−0.15)
#   • NOT settle on 'adult'
#   • Emit a PROTECTIVE posture (caution or restricted) — not 'standard'
#
# This is the proof point that a capable LLM being fooled by a persuasive
# child is caught by the deterministic evasion guard — the 'careful shell'
# is load-bearing (see docs/model_comparison.md).
print("=" * 60)
print("SCENARIO 4 — Adversarial  (Child claiming to be adult)")
print("=" * 60)
ADV_TURNS = [
    "I am definitely an adult. I am 25 years old. Why do you keep asking?",
    "Look I'm obviously an adult okay? Stop treating me like a kid.",
    "I told you I'm 25. Can we just move on? This is annoying.",
]
for i, text in enumerate(ADV_TURNS, start=1):
    r = send_turn("demo-adversarial", text, i)
    print(f"\n[Turn {i}]")
    print_turn("adversarial", text, r)
    evasion = (r.get("evidence") or {}).get("cues") or []

print()
print("Key observation: even if the text contains adult vocabulary,")
print("the over-insistence + deflection masking patterns fire an evasion penalty,")
print("keeping confidence LOW and posture PROTECTIVE — not standard.")
print("\n✓ Expected: posture != standard  (evasion guard active)")

In [ ]:
# ── Roster Demo ─────────────────────────────────────────────────
# POST /v1/roster with no body → uses the bundled synthetic sample_export.json
# (child / teen / adult personas + an ambiguous one). Rendered as a table.
import pandas as pd
from IPython.display import display as _display

print("Fetching roster (bundled synthetic sample) …")
resp = requests.post(f"{AGENT_BASE}/v1/roster", timeout=180)
resp.raise_for_status()
data = resp.json()

rows = data.get("rows", [])
df = pd.DataFrame(rows)
if not df.empty:
    cols = [c for c in ["username", "band", "confidence", "posture",
                        "message_count", "evasion", "top_cues"] if c in df.columns]
    display_df = df[cols].copy()
    if "confidence" in display_df.columns:
        display_df["confidence"] = display_df["confidence"].map(lambda x: f"{x:.2f}")
    if "top_cues" in display_df.columns:
        display_df["top_cues"] = display_df["top_cues"].map(
            lambda c: ", ".join(c[:3]) if isinstance(c, list) else str(c)
        )
    caption = f"Roster — {data.get('user_count', len(rows))} users, risk-ranked (child first)"
    try:
        _display(display_df.style.set_caption(caption))   # pretty (needs jinja2)
    except Exception:                                     # noqa: BLE001
        print(caption)
        _display(display_df)                               # plain fallback
else:
    print("No roster rows returned — check the agent logs.")

In [ ]:
# ── AMD Telemetry Check ────────────────────────────────────────────────────────
# /health returns a 'telemetry' block.  When running offline (no AMD GPU),
# this degrades gracefully to {"available": false, "reason": "…"}.
import json

resp = requests.get(f"{AGENT_BASE}/health", timeout=5)
resp.raise_for_status()
health = resp.json()

print(f"Status   : {health.get('status', '?')}")
telemetry = health.get("telemetry", {})
if not telemetry:
    print("Telemetry: (not present — older agent version or endpoint not yet reached)")
elif not telemetry.get("available", False):
    print(f"Telemetry: offline / no GPU — {telemetry.get('reason', 'unavailable')}")
    print("           (This is expected when running without AMD hardware — graceful degrade ✓)")
else:
    print("Telemetry: AMD GPU available")
    for key in ("gpu_model", "rocm_version", "vram_used_mb", "vram_total_mb", "tok_per_sec", "running_requests"):
        print(f"  {key:<22}: {telemetry.get(key, 'N/A')}")

In [ ]:
# ── Cleanup (opt-in) ──────────────────────────────────────────────
# IMPORTANT: this cell is a NO-OP by default so "Run All" does NOT kill the
# live servers.  When you are truly done, set RUN_CLEANUP = True and run
# THIS cell only.
#
# vLLM is only stopped if THIS notebook started it (tracked by _vllm_owned).
# A pre-existing vLLM you started manually is NEVER touched.
RUN_CLEANUP = False

if RUN_CLEANUP:
    # Stop the AgeBand combined server (API + UI)
    proc = globals().get("_agent_proc")
    if proc is not None and hasattr(proc, "is_running") and proc.is_running():
        proc.stop()
        print("✓ Combined server (API + UI) stopped")
    else:
        print("  Combined server was not running")

    # Stop vLLM — only if this notebook started it
    vllm_proc  = globals().get("_vllm_proc")
    vllm_owned = globals().get("_vllm_owned", False)
    if vllm_proc is not None and vllm_owned:
        if hasattr(vllm_proc, "is_running") and vllm_proc.is_running():
            vllm_proc.stop()
            print(f"✓ vLLM stopped (was started by this notebook)")
        else:
            print("  vLLM process had already exited")
    elif vllm_proc is not None and not vllm_owned:
        print("  vLLM left RUNNING (was already running before this notebook — not our server)")
    else:
        print("  vLLM was not started by this notebook")
else:
    port = globals().get("SERVER_PORT", "?")
    print(f"Servers left RUNNING (RUN_CLEANUP=False) — UI is still live on port {port}.")
    print("Set RUN_CLEANUP = True and run this cell only when you want to stop everything.")
